# Feature importance analysis
In this laboratory , you will use a Random Forest to evaluate the relative importance of the features of the training set. This technique is often used to get rid of irrelevant features before training. The network traffic has been previously pre-processed in a way that packets are grouped in bi-directional traffic flows using the 5-tuple (source IP, destination IP, source Port, destination Port, protocol). Each flow is represented with 21 packet-header statistical features computed with max 10 packets/flow:

- timestamp (mean IAT)
- packet_length (mean)
- IP_flags_df (sum)
- IP_flags_mf (sum)
- IP_flags_rb (sum)
- IP_frag_off (sum)
- protocols (mean)
- TCP_length (mean)
- TCP_flags_ack (sum)
- TCP_flags_cwr (sum)
- TCP_flags_ece (sum)
- TCP_flags_fin (sum)
- TCP_flags_push (sum)
- TCP_flags_res (sum)
- TCP_flags_reset (sum)
- TCP_flags_syn (sum)
- TCP_flags_urg (sum)
- TCP_window_size (mean)
- UDP_length (mean)
- ICMP_type (mean)
- Packets (counter)

In [ ]:
# Author: Roberto Doriguzzi-Corin
# Project: Course on Network Intrusion and Anomaly Detection with Machine Learning
#
# Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
#   http://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.

import os
import numpy as np
import glob
import h5py
import sys
import copy
import argparse
from sklearn.metrics import classification_report, accuracy_score
import logging
from util_functions import *
from IPython.display import Image, display
from sklearn.tree import export_graphviz
from sklearn.ensemble import RandomForestClassifier

OUTPUT_FILE = "./rf_tree"
DATASET_FOLDER = "./DOS2019"
X_train, y_train = load_dataset(DATASET_FOLDER + "/*" + '-train.hdf5')

from matplotlib import pyplot as plt
plt.rcParams.update({'figure.figsize': (12.0, 8.0)})
plt.rcParams.update({'font.size': 14})

SEED=1
feature_names = get_feature_names()
target_names = ['benign', 'dns',  'syn', 'udplag', 'webddos'] #IMPORTANT: when adding new classes, maintain the alphabetical order
target_names_full = ['benign', 'dns', 'ldap', 'mssql', 'netbios', 'ntp', 'portmap', 'snmp', 'ssdp', 'syn', 'tftp', 'udp', 'udplag', 'webddos'] # we use this to match class names with the class numbers returned by the RF
X_train, y_train = load_dataset(DATASET_FOLDER + "/*" + '-train.hdf5')
y_train = np.where(y_train==1)[1] 

In [ ]:
def show_tree(tree_clf, feature_names):
    export_graphviz(
        tree_clf,
        out_file=OUTPUT_FILE + ".dot",
        feature_names=feature_names,
        class_names=target_names,
        rounded=True,
        filled=True
    )

    # comvert the "dot" file into a png image
    os.system("dot -Tpng " + OUTPUT_FILE + ".dot -o " + OUTPUT_FILE + ".png")
    display(Image(filename=OUTPUT_FILE + ".png"))

# Classification with Random Forests
Implement a Random Forest classifier with 100 trees (estimators) and play with the regularisation hyper-parameters, such as max_depth, min_samples_split, min_samples_leaf.
Replace the RF Classifier with an ExtraTreesClassifier and test the regularisation hyper-parameters

In [ ]:
### Put you code here: define the RF model and train it using the dataset loaded above
### Set the number of estimators (n_estimators), the stopping strategy (e.g., max_depth) and enable the oob_score=True
rf = 

###

# Validation using the OOB score
The "OOB score" stands for "Out-of-Bag score," and it is a metric used in the context of random forests for estimating the model's performance on unseen data **without the need for a separate validation set**. It's a valuable tool for assessing the generalization capability of a random forest classifier or regressor.

In [ ]:
### Play the RF hyperparameters of the RF model to see what configuration works best for this problem
oob_score = 

print(rf.get_params())
print ("Accuracy score: ", oob_score)

In [ ]:
# Let's visualise some decision tree of the Random Forest
tree_clf = rf.estimators_[0]
show_tree(tree_clf, feature_names)

# Feature importance
Let's now plot the most important features, as computed using the average decrease of the Gini impurity.

In [ ]:
### Assign the feature importances to "fi"
fi = 
plt.barh(feature_names, fi)
plt.show()

# Inference using the RF model
Use the trained RF to make prediction on the test set. 

In [ ]:
X_test, y_test = load_dataset(DATASET_FOLDER + "/*" + '-test.hdf5')
y_test = np.where(y_test==1)[1] #from one-shot-encoding to numbers

### Replace the three dots with your code
y_pred = ...
print(classification_report(y_test, y_pred, target_names=target_names))

# Importance with feature reset
Use the above code to implement a feature ranking algorithm by resetting one feature at a time.
In this case, you might obtain negative values for some features. This means that those features can be noisy, irrelevant or redundant.

In [ ]:
import copy
X_train, y_train = load_dataset(DATASET_FOLDER + "/*" + '-train.hdf5')
y_train = np.where(y_train==1)[1] 
X_test, y_test = load_dataset(DATASET_FOLDER + "/*" + '-test.hdf5')
y_test = np.where(y_test==1)[1] 

### Instanciate and traing a RF classifier
rf = 

### Predict with the trained model
y_pred = 

### Here we save the accuracy score that we use as a baseline for evaluating the importance of each feature
baseline_accuracy = accuracy_score(y_test,y_pred)

feature_ranking = {}
for feature_index in range(len(feature_names)):
    X_test_redux = copy.deepcopy(X_test)
    ### Set one to zero one feature at a time using the feature_index and the np.zeros method
    X_test_redux[:,feature_index] = 

    ### Predict using the new test set with one feature set to zero
    y_pred_redux = 

    # Here we save the difference in accuracy with the baseline
    feature_ranking[feature_names[feature_index]] = baseline_accuracy - accuracy_score(y_test,y_pred_redux)

print (feature_ranking)
plt.barh(feature_names, feature_ranking.values())
plt.show()